In [2]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from collections import deque

In [ ]:
env = gym.make("CartPole-v1")

state, info = env.reset()

print("Initial state:", state)
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)

In [ ]:
seed = 42

np.random.seed(seed)
torch.manual_seed(seed)

rng = np.random.default_rng(seed)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if device.type == "cuda":
    torch.cuda.manual_seed_all(seed)

print(f"Using device: {device}")

env.action_space.seed(seed)

num_episodes = 600

gamma = 0.99
learning_rate = 1e-3

batch_size = 128

replay_buffer_size = 50_000
minimum_replay_size = 1_000

target_update_frequency = 500

epsilon_init = 1.0
epsilon_final = 0.05
epsilon_decay = 10_000

In [ ]:
class DQN(nn.Module):

    def __init__(self, state_dim, action_dim):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(state_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 128),
            nn.ReLU(),

            nn.Linear(128, action_dim)
        )

    def forward(self, state):

        return self.network(state)

In [ ]:
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

policy_network = DQN(state_dim, action_dim).to(device)
target_network = DQN(state_dim, action_dim).to(device)

target_network.load_state_dict(
    policy_network.state_dict()
)

target_network.eval()

In [ ]:
optimizer = optim.Adam(
    policy_network.parameters(),
    lr=learning_rate
)

loss_function = nn.SmoothL1Loss()

In [ ]:
replay_buffer = deque(
    maxlen=replay_buffer_size
)

In [ ]:
def get_epsilon(step):

    epsilon = (
        epsilon_final
        +
        (epsilon_init - epsilon_final)
        * np.exp(-step / epsilon_decay)
    )

    return epsilon

In [ ]:
def select_action(state, epsilon):

    # Exploration
    if rng.random() < epsilon:

        action = rng.integers(
            env.action_space.n
        )

        return int(action)

    # Exploitation
    state_tensor = torch.tensor(
        state,
        dtype=torch.float32,
        device=device
    ).unsqueeze(0)

    with torch.no_grad():

        q_values = policy_network(
            state_tensor
        )

    action = torch.argmax(
        q_values,
        dim=1
    ).item()

    return action

In [ ]:
def train_dqn():

    if len(replay_buffer) < minimum_replay_size:
        return None

    indices = rng.choice(
        len(replay_buffer),
        size=batch_size,
        replace=False
    )

    batch = [
        replay_buffer[i]
        for i in indices
    ]

    states = np.array([
        transition[0]
        for transition in batch
    ])

    actions = np.array([
        transition[1]
        for transition in batch
    ])

    rewards = np.array([
        transition[2]
        for transition in batch
    ])

    next_states = np.array([
        transition[3]
        for transition in batch
    ])

    terminated = np.array([
        transition[4]
        for transition in batch
    ])

    states = torch.tensor(
        states,
        dtype=torch.float32,
        device=device
    )

    actions = torch.tensor(
        actions,
        dtype=torch.long,
        device=device
    )

    rewards = torch.tensor(
        rewards,
        dtype=torch.float32,
        device=device
    )

    next_states = torch.tensor(
        next_states,
        dtype=torch.float32,
        device=device
    )

    terminated = torch.tensor(
        terminated,
        dtype=torch.float32,
        device=device
    )

    # Current Q(s,a)
    q_values = policy_network(states)

    current_q_values = q_values.gather(
        1,
        actions.unsqueeze(1)
    ).squeeze(1)

    # DQN target
    with torch.no_grad():

        next_q_values = target_network(
            next_states
        )

        max_next_q_values = (
            next_q_values.max(dim=1).values
        )

        target_q_values = (
            rewards
            +
            gamma
            * (1 - terminated)
            * max_next_q_values
        )

    loss = loss_function(
        current_q_values,
        target_q_values
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    return loss.item()

In [ ]:
episode_rewards = []
episode_losses = []

total_steps = 0

print(f"Starting DQN training for up to {num_episodes} episodes...")

for episode in range(num_episodes):

    state, info = env.reset(
        seed=seed + episode
    )

    episode_reward = 0

    episode_loss_values = []

    terminated = False
    truncated = False

    while not (terminated or truncated):

        epsilon = get_epsilon(
            total_steps
        )

        action = select_action(
            state,
            epsilon
        )

        next_state, reward, terminated, truncated, info = \
            env.step(action)

        replay_buffer.append(
            (
                state,
                action,
                reward,
                next_state,
                terminated
            )
        )

        state = next_state

        episode_reward += reward

        total_steps += 1

        loss = train_dqn()

        if loss is not None:

            episode_loss_values.append(
                loss
            )

        # Update target network
        if (
            total_steps
            % target_update_frequency
            == 0
        ):

            target_network.load_state_dict(
                policy_network.state_dict()
            )

    episode_rewards.append(
        episode_reward
    )

    if episode_loss_values:

        episode_losses.append(
            np.mean(episode_loss_values)
        )

    else:

        episode_losses.append(
            np.nan
        )

    if (episode + 1) % (int(num_episodes/20)) == 0:

        average_reward = np.mean(
            episode_rewards[-10:]
        )

        mean_loss = np.nanmean(episode_losses[-10:])

        print(
            f"Episode: {episode + 1:4d}/{num_episodes} | "
            f"Reward: {episode_reward:6.1f} | "
            f"Avg reward (10): {average_reward:6.1f} | "
            f"Avg loss (10): {mean_loss:.4f} | "
            f"Epsilon: {epsilon:.3f} | "
            f"Steps: {total_steps}"
        )

    # CartPole-v1 is considered essentially solved
    # when the 100-episode mean approaches 475+
    if len(episode_rewards) >= 100:

        average_100 = np.mean(
            episode_rewards[-100:]
        )

        if average_100 >= 475:

            print(
                "\nEnvironment solved!"
            )

            print(
                f"Episode: {episode + 1}"
            )

            print(
                f"Average reward over "
                f"last 100 episodes: "
                f"{average_100:.2f}"
            )

            break

print("\nDQN training loop complete.")
print("Episodes trained:", len(episode_rewards))
print("Total environment steps:", total_steps)

In [ ]:
# Notify the user immediately after neural-network training finishes.
from utils import alert_training_complete

alert_training_complete(
    "Training finished - the CartPole DQN is ready."
)

In [ ]:
env.close()

In [ ]:
episode_rewards_array = np.array(
    episode_rewards
)

window = 20

moving_average = np.convolve(
    episode_rewards_array,
    np.ones(window) / window,
    mode="valid"
)

plt.figure(figsize=(10, 5))

plt.plot(
    episode_rewards,
    alpha=0.4,
    label="Episode reward"
)

plt.plot(
    np.arange(window - 1, len(episode_rewards)),
    moving_average,
    label="20-episode moving average"
)

plt.xlabel("Episode")
plt.ylabel("Reward")
plt.title("CartPole DQN Training")
plt.legend()

plt.show()

In [ ]:
test_env = gym.make(
    "CartPole-v1"
)

num_test_episodes = 100

test_rewards = []

for episode in range(num_test_episodes):

    state, info = test_env.reset()

    terminated = False
    truncated = False

    total_reward = 0

    while not (terminated or truncated):

        state_tensor = torch.tensor(
            state,
            dtype=torch.float32,
            device=device
        ).unsqueeze(0)

        with torch.no_grad():

            q_values = policy_network(
                state_tensor
            )

        action = torch.argmax(
            q_values,
            dim=1
        ).item()

        state, reward, terminated, truncated, info = \
            test_env.step(action)

        total_reward += reward

    test_rewards.append(
        total_reward
    )

test_env.close()

In [ ]:
print(
    f"Average test reward: "
    f"{np.mean(test_rewards):.2f}"
)

print(
    f"Minimum test reward: "
    f"{np.min(test_rewards):.2f}"
)

print(
    f"Maximum test reward: "
    f"{np.max(test_rewards):.2f}"
)

In [ ]:
render_env = gym.make(
    "CartPole-v1",
    render_mode="human"
)

state, info = render_env.reset()

terminated = False
truncated = False

while not (terminated or truncated):

    state_tensor = torch.tensor(
        state,
        dtype=torch.float32,
        device=device
    ).unsqueeze(0)

    with torch.no_grad():

        q_values = policy_network(
            state_tensor
        )

    action = torch.argmax(
        q_values,
        dim=1
    ).item()

    state, reward, terminated, truncated, info = \
        render_env.step(action)

render_env.close()

In [ ]:
from pathlib import Path

model_path = Path("models") / "cartpole_dqn.pth"
model_path.parent.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "model_state_dict":
            policy_network.state_dict(),

        "gamma":
            gamma,

        "learning_rate":
            learning_rate,

        "epsilon_init":
            epsilon_init,

        "epsilon_final":
            epsilon_final,

        "epsilon_decay":
            epsilon_decay,

        "episodes_trained":
            len(episode_rewards),

        "environment":
            "CartPole-v1"
    },

    model_path
)

print(f"Saved trained DQN model to: {model_path}")

In [ ]:
checkpoint = torch.load(
    Path("models") / "cartpole_dqn.pth",
    map_location=device
)

policy_network.load_state_dict(
    checkpoint["model_state_dict"]
)

policy_network.eval()